<a href="https://colab.research.google.com/github/41371204h/1142-programming-language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q google-generativeai

In [11]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [12]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [13]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make smart decisions.


In [15]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1bMteJ88u-_o-6v1Ym1RZNDrdIpQA3_A6OucFpDQbyBs/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [16]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [24]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        # 將回傳內容中的 * 替換為空白
        summary = response.text.replace('*', ' ')
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [28]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 5. 將 AI 摘要整合成「一列」資料並寫入
        # 格式：[日期, "AI 摘要", 完整摘要內容]
        summary_row = [
            datetime.now().strftime('%Y-%m-%d'),
            'AI 摘要',
            summary
        ]

        # 使用 append_row 確保它永遠出現在最後一行，不覆蓋舊資料
        ws.append_row(summary_row)

        print("\n--- AI 摘要已成功寫入 Google Sheet 最後一行。---")
        print("內容摘要：\n", summary)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：80
已記錄：日期: 2026-04-02, 科目: 國文, 成績: 80

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：95
已記錄：日期: 2026-04-02, 科目: 數學, 成績: 95

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：92
已記錄：日期: 2026-04-02, 科目: 英文, 成績: 92

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet 最後一行。---
內容摘要：
 好的，這是一份根據您提供的學生單次成績紀錄所做的摘要與常見迷思整理，全程不對學生成績進行評價或好壞判斷，僅做客觀陳述與一般性迷思探討。

---

###   學生單次成績紀錄摘要  

這份成績紀錄顯示該學生在2026年4月2日參與了國文、數學和英文三科的測驗。從結果來看，學生的表現橫跨80分至95分之間。其中，數學科目取得了最高分95分，英文為92分，而國文則獲得80分。整體而言，該次測驗的平均分數約為89分。

---

###   關於成績的常見迷思整理（不評分，只做總結）  

當我們看到一份成績單時，很容易會產生一些既定的看法或聯想。以下整理一些關於成績的常見迷思，並提供更全面的視角：

1.    迷思一：單次成績完全決定學生的學習能力或智商。  
          澄清：   成績單僅是學生在特定時間點、針對特定測驗內容所展現的表現。它可能受到多種因素影響，包括：考試範圍、題型、學生當下的身體狀況、心理壓力、甚至閱讀理解試題的方式等。單次成績無法全面衡量學生的潛力、學習過程中的努力、解決問題的能力、創意或多元智能。

2.    迷思二：不同科目的分數可以直接比較優劣。  
          澄清：   國文、數學、英文等科目本質上涉及不同的知識體系、思維模式和評量標準。例如，數學可能更注重邏輯推演和精確計算，而國文可能側重語感、表達和文化理解。單純比較數字高低，忽略科目特性與評量重點，